# BEE 4750 Homework 5: Mixed Integer and Stochastic Programming

**Name**:

**ID**:

> **Due Date**
>
> Thursday, 12/04/24, 9:00pm

## Overview

### Instructions

-   In Problem 1, you will use mixed integer programming to solve a
    waste load allocation problem.
-   In Problem 2, you will formulate a stochastic optimization problem.

### Load Environment

The following code loads the environment and makes sure all needed
packages are installed. This should be at the start of most Julia
scripts.

In [51]:
import Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()

  Activating project at `c:\Users\nicho\OneDrive\Documents\hw05`
┌ Warning: The project dependencies or compat requirements have changed since the manifest was last resolved.
│ It is recommended to `Pkg.resolve()` or consider `Pkg.update()` if necessary.
└ @ Pkg.API C:\Users\nicho\.julia\juliaup\julia-1.12.1+0.x64.w64.mingw32\share\julia\stdlib\v1.12\Pkg\src\API.jl:1227
Precompiling packages...
              ✗ MbedTLS
              ✗ HTTP
  0 dependencies successfully precompiled in 4 seconds. 249 already precompiled.


In [52]:
using JuMP
using HiGHS
using DataFrames
using GraphRecipes
using Plots
using Measures
using MarkdownTables

## Problems (Total: 30 Points)

### Problem 1 (24 points)

Three cities are developing a coordinated municipal solid waste (MSW)
disposal plan. Three disposal alternatives are being considered: a
landfill (LF), a materials recycling facility (MRF), and a
waste-to-energy facility (WTE). The capacities of these facilities and
the fees for operation and disposal are provided below.

-   **LF**: Capacity 200 Mg, fixed cost \$2000/day, tipping cost
    \$50/Mg;
-   **MRF**: Capacity 350 Mg, fixed cost \$1500/day, tipping cost
    \$7/Mg, recycling cost \$40/Mg recycled;
-   **WTE**: Capacity 210 Mg, fixed cost \$2500/day, tipping cost
    \$60/Mg;

Transportation costs are
\$1.5/Mg-km, and the relative distances between the cities and
facilities are provided in the table below.

| **City/Facility** | **Landfill (km)** | **MRF (km)** | **WTE (km)** |
|:-----------------:|:-----------------:|:------------:|:------------:|
|         1         |         5         |      30      |      15      |
|         2         |        15         |      25      |      10      |
|         3         |        13         |      45      |      20      |
|        LF         |        \-         |      32      |      18      |
|        MRF        |        32         |      \-      |      15      |
|        WTE        |        18         |      15      |      \-      |

The fixed costs associated with the disposal options are incurred only
if the particular disposal option is implemented. The three cities
produce 100, 90, and 120 Mg/day of solid waste, respectively, with the
composition provided in the table below.

| **Component** | **% of total mass** | **Combustion ash** (%) | **MRF Recycling rate** (%) |
|:---------------------:|:--------------:|:---------------:|:---------------:|
| Food Wastes | 15 | 8 | 0 |
| Paper & Cardboard | 40 | 7 | 55 |
| Plastics | 5 | 5 | 15 |
| Textiles | 3 | 10 | 10 |
| Rubber, Leather | 2 | 15 | 0 |
| Wood | 5 | 2 | 30 |
| Yard Wastes | 18 | 2 | 40 |
| Glass | 4 | 100 | 60 |
| Ferrous | 2 | 100 | 75 |
| Aluminum | 2 | 100 | 80 |
| Other Metal | 1 | 100 | 50 |
| Miscellaneous | 3 | 70 | 0 |

The information in the above table will help you determine the overall
recycling and ash fractions. Note that the recycling residuals, which
may be sent to either landfill or the WTE, have different ash content
than the ash content of the original MSW. You will need to determine
these fractions to construct your mass balance constraints.

**Reminder**: Use `round(x; digits=n)` to report values to the
appropriate precision!

#### Problem 1.1

Based on the information above, calculate the overall recycling and ash
fractions for the waste produced by each city.

Problem 1.1


Overall recycling: 37.75 percent, this was found from calculating the product of each component's total mass and recycling rate and then adding up the sum of all of the products.

Using the overall recycle fraction of 37.75, we get the following for each city:

City 1: 100 Mg/day * .3775 = 37.75 Mg/day

City 2: 90 Mg/day * .3775 = 33.98 Mg/day

City 3: 120 Mg/day * .3775 = 45.30 Mg/day

Ash fraction = 16.41 percent, this was found from calculating the product of each component's total mass and combusting ash and then adding up the sum of all of the products.

City 1: 100 Mg/day * .1641 = 16.41 Mg/day

City 2: 90 Mg/day * .1641 = 14.77 Mg/day

City 3: 120 Mg/day * .1641 = 19.69 Mg/day

Also worth noting is the ash fraction of the residual stream. We can begin calculating this by finding the residual mass fraction which is just 100 - 37.75, so 62.25.  We can then find the total residual ash by taking the sum of the residual mass for each component times the ash fraction for each component.  Finding the sum of all of these products for the components gives 8.9345%.  We then divide 8.9345 by our total residual mass which is 62.25 to get 14.35%.

#### Problem 1.2

What are the decision variables for your optimization problem? Provide
notation and variable meaning.

In [53]:
msw_model = Model(HiGHS.Optimizer) #initialize model object

#variable for waste flows between 3 cities and 3 facilities
@variable(msw_model, quantity[1:3, 1:3]) 

#variable for if facilitiy is operated or not (binary)
@variable(msw_model, y[1:3], Bin)

#variable for residual waste from mrf to lf
@variable(msw_model, res_LF)

#variable for residual waste from mrf to wte
@variable(msw_model, res_WTE)

res_WTE

#### Problem 1.3

Formulate the objective function. Make sure to include any needed
derivations or justifications for your equation(s).

In [54]:
msw_model = Model(HiGHS.Optimizer) #initialize model object

#variable for waste flows between 3 cities and 3 facilities
@variable(msw_model, quantity[1:3, 1:3] >= 0) 

#variable for if facility is operated or not
@variable(msw_model, y[1:3], Bin)

#variable for residual waste from mrf to lf
@variable(msw_model, res_LF >= 0)

#variable for residual waste from mrf to wte
@variable(msw_model, res_WTE >= 0)

#Establish constants

recycle = .3775
ash_msw = 0.1641
ash_residual = 0.1435

mrf_recycle = 40
transport = 1.5
cost_fixed_1 = 2000
cost_fixed_2 = 1500
cost_fixed_3 = 2500
tipping_1 = 50
tipping_2 = 7
tipping_3 = 60

city_distances = [
5.0 30.0 15.0
15.0 25.0 10.0
13.0 45.0 20.0
]

distance_lf_mrf = 32 
distance_lf_wte = 18
distance_wte_mrf = 15

#expressions for total flow to MRF and recycled amount at MRF

@expression(msw_model, total_flow_MRF, sum(quantity[i,2] for i in 1:3))
@expression(msw_model, total_recycled, recycle * total_flow_MRF)

#Establish expressions for readability
#for all of the different cost variables

#sums the fixed costs for each facility
@expression(msw_model, fixed_costs, cost_fixed_1 * y[1] + 
cost_fixed_2 * y[2] + cost_fixed_3 * y[3])

#calculates total tipping cost based on waste quantity, tipping costs, and residual waste
@expression(msw_model, tipping_cost, tipping_1 * (sum(quantity[i,1] for i in 1:3) + res_LF) 
+ tipping_2 * (sum(quantity[i,2] for i in 1:3)) + tipping_3 * (sum(quantity[i,3] for i in 1:3) + res_WTE))

#calculates total recycling cost with mrf facility and total
@expression(msw_model, recycling_cost, mrf_recycle * total_recycled)

#calculates transport cost based on waste amount, city distances, transport costs, and residual waste
@expression(msw_model, transport_cost,  (transport * (sum(quantity[i,j] * city_distances[i,j] for i in 1:3, j in 1:3) 
        + res_LF * distance_lf_mrf + res_WTE * distance_wte_mrf)))

#Expression to get total cost to call in objective function
@expression(msw_model, total_cost, fixed_costs + tipping_cost + recycling_cost + transport_cost)

#Minimize total cost
@objective(msw_model, Min, total_cost)

2000 y[1] + 1500 y[2] + 2500 y[3] + 57.5 quantity[1,1] + 72.5 quantity[2,1] + 69.5 quantity[3,1] + 98 res_LF + 67.1 quantity[1,2] + 59.6 quantity[2,2] + 89.6 quantity[3,2] + 82.5 quantity[1,3] + 75 quantity[2,3] + 90 quantity[3,3] + 82.5 res_WTE

#### Problem 1.4

Derive all relevant constraints. Make sure to include any needed
justifications or derivations.

In [55]:
#Supply constraint
#we know quantity of waste has to add up to match the waste produced 
#by the cities

supply = [100, 90, 120]

for i in 1:3
    @constraint(msw_model, sum(quantity[i, j] for j in 1:3) == supply[i])
end

#Residual source constraint

@constraint(msw_model, res_LF + res_WTE == (1 - recycle) *
    sum(quantity[i, 2] for i in 1:3))

#Capacity constraints

capacity_1 = 200
capacity_2 = 350
capacity_3 = 210

#calculates ash amount from the WTE facility
@expression(msw_model, ash_from_WTE, ash_msw * sum(quantity[i,3] for i in 1:3) + ash_residual * res_WTE)

#Constraints capacities for each facility
@constraint(msw_model, sum(quantity[i,1] for i in 1:3) + res_LF + ash_from_WTE <= capacity_1 * y[1])  
@constraint(msw_model, sum(quantity[i,2] for i in 1:3) <= capacity_2 * y[2])                            
@constraint(msw_model, sum(quantity[i,3] for i in 1:3) + res_WTE <= capacity_3 * y[3])        


quantity[1,3] + quantity[2,3] + quantity[3,3] - 210 y[3] + res_WTE <= 0

#### Problem 1.5

Find the optimal solution (using `JuMP` to solve the problem). Report
the optimal objective value.

In [56]:
msw_model = Model(HiGHS.Optimizer) #initialize model object

#variable for waste flows between 3 cities and 3 facilities
@variable(msw_model, quantity[1:3, 1:3] >= 0) 

#variable for if facilitiy is operated or not
@variable(msw_model, y[1:3], Bin)

#variable for residual waste from mrf to lf
@variable(msw_model, res_LF >= 0)

#variable for residual waste from mrf to wte
@variable(msw_model, res_WTE >= 0)

#Establish constants

recycle = .3775
ash_msw = 0.1641
ash_residual = 0.1435

mrf_recycle = 40
transport = 1.5
cost_fixed_1 = 2000
cost_fixed_2 = 1500
cost_fixed_3 = 2500
tipping_1 = 50
tipping_2 = 7
tipping_3 = 60

city_distances = [
5.0 30.0 15.0
15.0 25.0 10.0
13.0 45.0 20.0
]

distance_lf_mrf = 32 
distance_lf_wte = 18
distance_wte_mrf = 15

#expressions for total flow to MRF and recycled amount at MRF

@expression(msw_model, total_flow_MRF, sum(quantity[i,2] for i in 1:3))
@expression(msw_model, total_recycled, recycle * total_flow_MRF)

#Establish expressions for readability
#for all of the different cost variables

@expression(msw_model, fixed_costs, cost_fixed_1 * y[1] + 
cost_fixed_2 * y[2] + cost_fixed_3 * y[3])

@expression(msw_model, tipping_cost, tipping_1 * (sum(quantity[i,1] for i in 1:3) + res_LF) 
+ tipping_2 * (sum(quantity[i,2] for i in 1:3)) + tipping_3 * (sum(quantity[i,3] for i in 1:3) + res_WTE))

@expression(msw_model, recycling_cost, mrf_recycle * total_recycled)

@expression(msw_model, transport_cost,  (transport * (sum(quantity[i,j] * city_distances[i,j] for i in 1:3, j in 1:3) 
        + res_LF * distance_lf_mrf + res_WTE * distance_wte_mrf)))

#Expression to get total cost to call in objective function
@expression(msw_model, total_cost, fixed_costs + tipping_cost + recycling_cost + transport_cost)

#Minimize total cost
@objective(msw_model, Min, total_cost)

#Supply constraint

supply = [100, 90, 120]

for i in 1:3
    @constraint(msw_model, sum(quantity[i, j] for j in 1:3) == supply[i])
end

#Residual source constraint

@constraint(msw_model, res_LF + res_WTE == (1 - recycle) *
    sum(quantity[i, 2] for i in 1:3))

#Capacity constraints

capacity_1 = 200
capacity_2 = 350
capacity_3 = 210

@expression(msw_model, ash_from_WTE, ash_msw * sum(quantity[i,3] for i in 1:3) + ash_residual * res_WTE)

@constraint(msw_model, sum(quantity[i,1] for i in 1:3) + res_LF + ash_from_WTE <= capacity_1 * y[1])  
@constraint(msw_model, sum(quantity[i,2] for i in 1:3) <= capacity_2 * y[2])                            
@constraint(msw_model, sum(quantity[i,3] for i in 1:3) + res_WTE <= capacity_3 * y[3])        

#optimize model
optimize!(msw_model)

#print results code

println("Quantity flows (Mg/day):")
for i in 1:3
    for j in 1:3
        println("quantity[$i,$j] = ", value(quantity[i,j]))
    end
end

println("Facility activation y: ", value.(y))

println("res_LF = ", value(res_LF))
println("res_WTE = ", value(res_WTE))

println("The total cost is ", round(objective_value(msw_model); digits = 2))

Running HiGHS 1.12.0 (git hash: 755a8e027): Copyright (c) 2025 HiGHS under MIT licence terms
MIP has 7 rows; 14 cols; 32 nonzeros; 3 integer variables (3 binary)
Coefficient ranges:
  Matrix  [1e-01, 4e+02]
  Cost    [6e+01, 2e+03]
  Bound   [1e+00, 1e+00]
  RHS     [9e+01, 1e+02]
Presolving model
7 rows, 14 cols, 32 nonzeros  0s
7 rows, 13 cols, 31 nonzeros  0s
Presolve reductions: rows 7(-0); columns 13(-1); nonzeros 31(-1) 

Solving MIP model with:
   7 rows
   13 cols (2 binary, 0 integer, 0 implied int., 11 continuous, 0 domain fixed)
   31 nonzeros

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T => Evaluate node; U => Unbounded; X => User solution; Y => HiGHS solution;
     Z => ZI Round; l => Trivial lower; p => Trivial point; u => Trivial upper; z => Trivial zero

        Nodes      |    B&B Tree     |            Objective 

#### Problem 1.6

Draw a diagram showing the flows of waste between the cities and the
facilities. Which facilities (if any) will not be used? Does this
solution make sense?

Problem 1.6

See diagram on google slides.

The MRF facility is not utilized.

This solution does make sense, especially given the 40$ per mg recycled cost.  It appears that activating this facility and incurring the fixed cost, does not reduce the overall total costs enough to justify the activation.  This also appears in the diagram with how there are no transport arrows between the different facilities.

### Problem 2 (6 points)

Consider a two-period economic dispatch problem, based on the
multi-period example from Lecture 14 (on 10/29). The generator data,
including ramping constraints for each generator, is provided in
\`data/generators.csv.’ In period 1, the demand is
$d_1 = 1100 \text{MW}$. In period 2, the demand is projected to be
$d_2 = 1200 \text{MW}$, but there is a 25% probability that it is \$1500
. In the first period, the solar capacity factor is $0.9$ and the wind
capacity factor is $0.45$, but in the second period, there is some
uncertainty: the forecasted solar and wind capacity factors are $0.95$
and $0.4$, respectively, but there is a 30% probability that they are
$0.75$ and $0.5$. Your goal is to identify how to dispatch your
generators to minimize the cost of meeting demand.

#### Problem 2.1

Draw a scenario tree for this problem.

#### Problem 2.2

Formulate a stochastic linear program for this problem based on your
scenario tree from Problem 2.1 and the data in `data/generators.csv`.

In [ ]:
gen_model = Model(HiGHS.Optimizer) #initialize model object

#Hard code data from generators.csv
#to be used in formulation code

plant_name = ["Biomass", "Hydroelectric", "Geothermal", "NG CCGT", "NG CT", "Wind", "Solar"]

situation_number = ["S1", "S2", "S3", "S4"]

Pmin = Dict("Biomass" => 0, "Hydroelectric" => 0, "Geothermal" => 0, "NG CCGT" => 220,
            "NG CT" => 100, "Wind" => 0, "Solar" => 0)

Pmax = Dict("Biomass" => 100, "Hydroelectric" => 500, "Geothermal" => 400, "NG CCGT" => 500,
            "NG CT" => 250, "Wind" => 300, "Solar" => 500)

VarCost = Dict("Biomass" => 5, "Hydroelectric" => 0, "Geothermal" => 0, "NG CCGT" => 23,
               "NG CT" => 38, "Wind" => 0, "Solar" => 0)

Ramp = Dict("Biomass" => 100, "Hydroelectric" => 500, "Geothermal" => 400, "NG CCGT" => 100,
            "NG CT" => 200, "Wind" => 300, "Solar" => 500)

#capacity factors

solar_cap_1 = 0.90
wind_cap_1 = 0.45
solar_cap_2 = Dict("S1" => 0.95, "S2" => 0.75, "S3" => 0.95, "S4" => 0.75)
wind_cap_2 = Dict("S1" => 0.40, "S2" => 0.50, "S3" => 0.40, "S4" => 0.50)

#demand factors

demand_1 = 1100
demand_2 = Dict("S1" => 1200.0, "S2" => 1200.0, "S3" => 1500.0, "S4" => 1500.0)

#period 1 demand

@variable(gen_model, g1[g in plant_name], lower_bound = Pmin[g], upper_bound = Pmax[g])

#period 2 demand

@variable(gen_model, g2[g in plant_name, s in situation_number], lower_bound = Pmin[g], upper_bound = Pmax[g])

#set decision tree probabilities

probs = Dict("S1" => 0.525, "S2" => 0.225, "S3" => 0.175, "S4" => 0.075)

#period 1 cost
@expression(gen_model, cost_1, sum(VarCost[g] * g1[g] for g in plant_name))

#period 2 cost
@expression(gen_model, cost_2, sum(probs[s] * sum(VarCost[g] * g2[g,s] for g in plant_name) for s in situation_number))

#total cost that will be used in the objective function
@expression(gen_model, total_expected_cost, cost_1 + cost_2)

#objective function which is to minimize total cost
@objective(gen_model, Min, total_expected_cost)

#period 1 demand constraint that total is equal to demand_1

@constraint(gen_model, sum(g1[g] for g in plant_name) == demand_1)

#period 2 demand constraint that total is equal to demand_2

for s in situation_number
    @constraint(gen_model, sum(g2[g,s] for g in plant_name) == demand_2[s])
end

#capacity constraints

@constraint(gen_model, g1["Solar"] <= Pmax["Solar"] * solar_cap_1)
@constraint(gen_model, g1["Wind"] <= Pmax["Wind"]  * wind_cap_1)

for s in situation_number
    @constraint(gen_model, g2["Solar", s] <= Pmax["Solar"] * solar_cap_2[s])
    @constraint(gen_model, g2["Wind",  s] <= Pmax["Wind"]  * wind_cap_2[s])
end

#ramping up and down constraints

for g in plant_name, s in situation_number
    @constraint(gen_model, g2[g,s] - g1[g] <= Ramp[g]) 
    @constraint(gen_model, g1[g] - g2[g,s] <= Ramp[g])   
end

## References

OpenAI. (2024). ChatGPT (Sept 25 GPT-5 version) [Large language model].
https://chat.openai.com:  I utilized ChatGPT to help specifically with conceptualizing and implementing some of the Jump syntax and a few clarification questions with problem set up interpretation.  I did also utilize it for some julia-specific syntax questions, but it mainly revolved around properly utilizing Jump.  